In [ ]:
pip install mne torch-geometric

: 

In [ ]:


import os
from os.path import dirname, join as pjoin
import scipy as sp
import scipy.io as sio
from scipy import signal
import numpy as np
from matplotlib.pyplot import specgram
import matplotlib.pyplot as plt
import numpy as np
import scipy.signal as sig
import networkx as nx
import torch as torch
from scipy.signal import welch
from scipy.stats import entropy
from sklearn.feature_selection import mutual_info_classif
from scipy.integrate import simpson
from sklearn.model_selection import KFold
from torch_geometric.loader import DataLoader
from torch.nn import Linear
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, GATConv, GATv2Conv, GAT, GraphNorm
from torch_geometric.nn import global_mean_pool
from torch import nn
from tqdm import tqdm
from torch_geometric.data import Data



In [ ]:
# --- Core Libraries ---
import numpy as np
import scipy.io as sio
from scipy import signal as sig
from scipy.integrate import simpson
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
import networkx as nx
import os
from os.path import join as pjoin

# --- PyTorch & PyG Libraries ---
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATv2Conv, global_mean_pool, GraphNorm
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader

# --- Scikit-learn for Evaluation ---
from sklearn.model_selection import KFold
from sklearn.metrics import confusion_matrix, classification_report

# --- MNE for Topographical Plots (Crucial for Visualization) ---
import mne

# --- GPU Setup ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# --- Matplotlib plotting style ---
plt.style.use('seaborn-v0_8-whitegrid')
print("All libraries imported and device is set.")

In [ ]:
def load_bci_competition_data(data_dir, subject_number):
    """
    Load BCI Competition IV Dataset 2a and convert to the expected format.
    
    Args:
        data_dir: Directory containing the .mat files
        subject_number: Subject number (1-9)
    
    Returns:
        subject_data: Dictionary with 'L' and 'R' keys containing trial data
    """
    mat_fname = pjoin(data_dir, f'A0{subject_number}T.mat')
    mat_contents = sio.loadmat(mat_fname, squeeze_me=True, struct_as_record=False)
    data = mat_contents['data']
    
    left_trials = []
    right_trials = []
    
    for session in data:
        # We are only interested in the training sessions which have labels
        if 'X' in session._fieldnames:
             eeg_data = session.X
             trial_positions = session.trial
             trial_labels = session.y
             
             # Use only the 22 EEG channels
             eeg_data = eeg_data[:, :22]
             
             # Extract trials (4 seconds * 250 Hz = 1000 samples)
             trial_length = 1000
             
             for start_pos, label in zip(trial_positions, trial_labels):
                 start_sample = int(start_pos)
                 end_sample = start_sample + trial_length
                 
                 if end_sample <= eeg_data.shape[0]:
                     trial_data = eeg_data[start_sample:end_sample, :]
                     
                     if label == 1: # Left hand
                         left_trials.append(trial_data)
                     elif label == 2: # Right hand
                         right_trials.append(trial_data)
    
    # Convert to the expected format (list of numpy arrays)
    subject_data = {
        'L': left_trials,
        'R': right_trials
    }
    
    return subject_data

# --- Example Usage ---
# Note: Adjust the path to where your data is stored on Kaggle.
data_dir = '/kaggle/input/bci-competition-iv-data-sets-2a'
subject_1_data_raw = load_bci_competition_data(data_dir, 1)

print(f"Loaded data for Subject 1.")
print(f"Number of Left hand MI trials: {len(subject_1_data_raw['L'])}")
print(f"Number of Right hand MI trials: {len(subject_1_data_raw['R'])}")
print(f"Shape of a single trial: {subject_1_data_raw['L'][0].shape}")
print(f"(Time Samples, EEG Channels)")

In [ ]:
def bandpass(data: np.ndarray, edges: list[float], sample_rate: float, poles: int = 5):
    """Apply a bandpass filter to the data."""
    sos = sig.butter(poles, edges, 'bandpass', fs=sample_rate, output='sos')
    filtered_data = sig.sosfiltfilt(sos, data, axis=0)
    return filtered_data

# --- Create a copy for processing ---
subject_1_data_processed = {
    'L': [trial.copy() for trial in subject_1_data_raw['L']],
    'R': [trial.copy() for trial in subject_1_data_raw['R']]
}

# --- Apply Bandpass Filter ---
fs = 250  # Sampling frequency
freq_band = [8, 30] # Mu and Beta rhythm band
for class_key in subject_1_data_processed:
    for i in range(len(subject_1_data_processed[class_key])):
        subject_1_data_processed[class_key][i] = bandpass(
            subject_1_data_processed[class_key][i], freq_band, fs
        )

print("Applied bandpass filter (8-30 Hz) to all trials.")

# --- Visualization of Filtering ---
# Select one trial and one channel to visualize
trial_raw = subject_1_data_raw['L'][0]
trial_filtered = subject_1_data_processed['L'][0]
channel_idx = 8 # C3 electrode

# 1. Time Domain Plot
time = np.arange(trial_raw.shape[0]) / fs
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
ax1.plot(time, trial_raw[:, channel_idx], label='Raw EEG', color='red', alpha=0.8)
ax1.set_title(f'Time Domain Signal (Channel C3)')
ax1.set_ylabel('Amplitude (µV)')
ax1.legend()
ax2.plot(time, trial_filtered[:, channel_idx], label='Filtered (8-30Hz) EEG', color='blue')
ax2.set_xlabel('Time (s)')
ax2.set_ylabel('Amplitude (µV)')
ax2.legend()
plt.suptitle('Effect of Bandpass Filtering', fontsize=16)
plt.show()


# 2. Frequency Domain Plot (Power Spectral Density)
freqs_raw, psd_raw = sig.welch(trial_raw[:, channel_idx], fs)
freqs_filt, psd_filt = sig.welch(trial_filtered[:, channel_idx], fs)
fig, ax = plt.subplots(figsize=(12, 5))
ax.semilogy(freqs_raw, psd_raw, label='Raw EEG Spectrum', color='red', alpha=0.7)
ax.semilogy(freqs_filt, psd_filt, label='Filtered EEG Spectrum', color='blue', linewidth=2)
ax.axvspan(freq_band[0], freq_band[1], color='green', alpha=0.2, label='Band of Interest')
ax.set_title('Power Spectral Density (PSD)')
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('Power/Frequency (dB/Hz)')
ax.set_xlim(0, 50)
ax.legend()
plt.show()

In [ ]:
def plvfcn(eeg_data):
    """
    Calculates the Phase Locking Value (PLV) matrix for a single trial.
    """
    num_channels = eeg_data.shape[1]
    num_samples = eeg_data.shape[0]
    analytic_signal = sig.hilbert(eeg_data, axis=0)
    instantaneous_phase = np.angle(analytic_signal)
    
    plv_matrix = np.zeros((num_channels, num_channels))
    for i in range(num_channels):
        for j in range(i + 1, num_channels):
            phase_diff = instantaneous_phase[:, i] - instantaneous_phase[:, j]
            plv = np.abs(np.sum(np.exp(1j * phase_diff)) / num_samples)
            plv_matrix[i, j] = plv
            plv_matrix[j, i] = plv
            
    return plv_matrix

def compute_all_plv_matrices(filtered_data):
    """
    Computes PLV matrices for all trials in the dataset.
    """
    all_plv = {'L': [], 'R': []}
    for class_key in filtered_data:
        for trial_data in tqdm(filtered_data[class_key], desc=f'Computing PLV for {class_key} trials'):
            plv_matrix = plvfcn(trial_data)
            all_plv[class_key].append(plv_matrix)
    return all_plv

# --- Compute PLV for our processed data ---
subject_1_plv = compute_all_plv_matrices(subject_1_data_processed)

# --- Average PLV matrices for visualization ---
avg_plv_L = np.mean(subject_1_plv['L'], axis=0)
avg_plv_R = np.mean(subject_1_plv['R'], axis=0)

print(f"\nComputed PLV matrices. Average Left PLV matrix shape: {avg_plv_L.shape}")

# --- VISUALIZATION 1: PLV Matrix Heatmap ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
# Plotting code for heatmaps remains the same...
sns.heatmap(avg_plv_L, ax=ax1, cmap='viridis', vmin=0, vmax=np.max([avg_plv_L, avg_plv_R]))
ax1.set_title('Average PLV Matrix (Left Hand MI)')
ax1.set_xlabel('Channel Index'); ax1.set_ylabel('Channel Index')
sns.heatmap(avg_plv_R, ax=ax2, cmap='viridis', vmin=0, vmax=np.max([avg_plv_L, avg_plv_R]))
ax2.set_title('Average PLV Matrix (Right Hand MI)')
ax2.set_xlabel('Channel Index');
plt.suptitle('All-to-All Functional Connectivity', fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.95]); plt.show()


# --- VISUALIZATION 2: Topographical Connectivity Plot (Final Corrected Version) ---
ch_names = ['FC5', 'FC3', 'FC1', 'FCz', 'FC2', 'FC4', 'FC6', 'C5', 'C3', 'C1', 
            'Cz', 'C2', 'C4', 'C6', 'CP5', 'CP3', 'CP1', 'CPz', 'CP2', 'CP4', 'CP6', 'Fz']
info = mne.create_info(ch_names=ch_names, sfreq=fs, ch_types='eeg')
montage = mne.channels.make_standard_montage('standard_1005')
info.set_montage(montage)

# This is our new, robust plotting function
def plot_brain_connectivity(plv_matrix, ax, title, threshold=0.8):
    """
    Plots brain connectivity on a 2D sensor layout.
    """
    # Use mne.viz.plot_sensors to draw the channel locations on the given axes
    # This is the corrected line that fixes the TypeError
    mne.viz.plot_sensors(info, kind='topomap', ch_type='eeg', show_names=True, show=False)
    
    # Get the 2D sensor positions from the info object's montage
    ch_coords = info.get_montage().get_positions()['ch_pos']
    pos = np.array([ch_coords[ch] for ch in info['ch_names']])

    # Find connections above the threshold
    source, target = np.where(plv_matrix > threshold)
    
    # Plot the connections
    for i in range(len(source)):
        # We only plot each connection once
        if source[i] > target[i]:
            x1, y1 = pos[source[i], :2]
            x2, y2 = pos[target[i], :2]
            
            weight = plv_matrix[source[i], target[i]]
            
            ax.plot([x1, x2], [y1, y2], 
                    color='red', 
                    linewidth=0.5 + 2 * weight,
                    alpha=0.5 + 0.5 * weight)

    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_frame_on(False)


# Plotting the brain networks
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6), subplot_kw={'aspect': 'equal'})
plot_brain_connectivity(avg_plv_L, ax1, 'Connectivity during Left Hand MI')
plot_brain_connectivity(avg_plv_R, ax2, 'Connectivity during Right Hand MI')
plt.suptitle('Brain Network Visualization (PLV > 0.35)', fontsize=16)
plt.tight_layout(rect=[0, 0.05, 1, 0.95])
plt.show()


print("MONTAGE VISUALIZATION CODE DIDNT WORK HENCE THE GRAPH AND THE ELECTRODE PLACEMENT IS NOT SHOW TOGETHER")

In [ ]:
def bandpower(data, low, high, fs=250):
    """
    Calculate band power using Welch's method and Simpson's rule for integration.
    
    Args:
        data (1D array): Time-series data for a single channel.
        low (float): Lower frequency bound.
        high (float): Upper frequency bound.
        fs (int): Sampling frequency.
        
    Returns:
        power (float): The absolute power in the specified frequency band.
    """
    # Use a window of 1 second (250 samples) for Welch's method
    win_len = fs
    freqs, psd = sig.welch(data, fs, nperseg=min(win_len, len(data)))
    
    # Find the indices of frequencies within our band of interest
    idx_band = np.logical_and(freqs >= low, freqs <= high)
    
    # Frequency resolution
    freq_res = freqs[1] - freqs[0]
    
    # Compute power by integrating the PSD using Simpson's rule (more accurate)
    power = simpson(psd[idx_band], dx=freq_res)
    
    return power

def calculate_band_power_features(filtered_data):
    """
    Calculates band power features for all trials.
    """
    # Define frequency bands of 4 Hz width from 8 Hz to 40 Hz
    bands = list(range(8, 41, 4)) # [8, 12, 16, 20, 24, 28, 32, 36, 40]
    num_bands = len(bands) - 1
    
    features = {'L': [], 'R': []}
    
    for class_key in filtered_data:
        for trial_data in tqdm(filtered_data[class_key], desc=f'Calculating features for {class_key} trials'):
            num_channels = trial_data.shape[1]
            trial_features = np.zeros((num_channels, num_bands))
            
            for ch_idx in range(num_channels):
                for band_idx in range(num_bands):
                    low_freq = bands[band_idx]
                    high_freq = bands[band_idx+1]
                    trial_features[ch_idx, band_idx] = bandpower(trial_data[:, ch_idx], low_freq, high_freq)
            
            features[class_key].append(trial_features)
            
    return features, bands

# --- Calculate features and define bands ---
subject_1_features, freq_bands = calculate_band_power_features(subject_1_data_processed)
print(f"\nCalculated features. Shape of features for one trial: {subject_1_features['L'][0].shape}")
print(f"(Channels, Frequency Bands)")

# --- VISUALIZATION 1: Feature Heatmap ---
avg_feat_L = np.mean(subject_1_features['L'], axis=0)
avg_feat_R = np.mean(subject_1_features['R'], axis=0)
band_labels = [f'{freq_bands[i]}-{freq_bands[i+1]}Hz' for i in range(len(freq_bands)-1)]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 7))
sns.heatmap(avg_feat_L, yticklabels=ch_names, xticklabels=band_labels, cmap='plasma', ax=ax1)
ax1.set_title('Average Node Features (Left Hand MI)')
ax1.set_ylabel('EEG Channel')
ax1.set_xlabel('Frequency Band')

sns.heatmap(avg_feat_R, yticklabels=False, xticklabels=band_labels, cmap='plasma', ax=ax2)
ax2.set_title('Average Node Features (Right Hand MI)')
ax2.set_xlabel('Frequency Band')

plt.suptitle('Band Power as Node Features', fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

# --- VISUALIZATION 2: Topographical Power Map ---
# Let's visualize power in the Mu band (8-12 Hz), which is the first band (index 0)
mu_power_L = avg_feat_L[:, 0]
mu_power_R = avg_feat_R[:, 0]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))
mne.viz.plot_topomap(mu_power_L, info, axes=ax1, show=False, cmap='plasma')
ax1.set_title('Mu Band Power (8-12 Hz) for Left MI')
mne.viz.plot_topomap(mu_power_R, info, axes=ax2, show=False, cmap='plasma')
ax2.set_title('Mu Band Power (8-12 Hz) for Right MI')

# Add a colorbar
sm = plt.cm.ScalarMappable(cmap='plasma', norm=plt.Normalize(vmin=min(mu_power_L.min(), mu_power_R.min()), 
                                                            vmax=max(mu_power_L.max(), mu_power_R.max())))
cbar = fig.colorbar(sm, ax=[ax1, ax2], orientation='vertical', fraction=0.02, pad=0.04)
cbar.set_label('Power (µV²/Hz)')

plt.suptitle('Topographical Distribution of Node Features', fontsize=16)
plt.show()

In [ ]:
from torch_geometric.utils import dense_to_sparse

# --- 1. Assemble Data into PyTorch Geometric's format ---

# We will create a fully connected graph for each trial initially.
# The edge_weights will be the PLV values. The GAT's attention
# mechanism will then learn which of these connections are important.

data_list = []
# Process Left Hand trials (label 0)
for i in range(len(subject_1_plv['L'])):
    # Node features: [num_channels, num_features]
    x = torch.tensor(subject_1_features['L'][i], dtype=torch.float)
    
    # Adjacency matrix: [num_channels, num_channels]
    adj = torch.tensor(subject_1_plv['L'][i], dtype=torch.float)
    
    # Convert dense adjacency matrix to sparse edge_index and edge_attr
    edge_index, edge_attr = dense_to_sparse(adj)
    
    # Label
    y = torch.tensor([0], dtype=torch.long)
    
    data_list.append(Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y))

# Process Right Hand trials (label 1)
for i in range(len(subject_1_plv['R'])):
    x = torch.tensor(subject_1_features['R'][i], dtype=torch.float)
    adj = torch.tensor(subject_1_plv['R'][i], dtype=torch.float)
    edge_index, edge_attr = dense_to_sparse(adj)
    y = torch.tensor([1], dtype=torch.long)
    
    data_list.append(Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y))

print(f"Total number of graphs (trials) created: {len(data_list)}")
print("\n--- Example Data Object ---")
print(data_list[0])
print("---------------------------")


# --- 2. Define the GAT Model Architecture ---

class GAT(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, heads):
        super(GAT, self).__init__()
        # We use GATv2Conv as it's a more powerful and stable version
        self.conv1 = GATv2Conv(in_channels, hidden_channels, heads=heads, concat=True, edge_dim=1)
        self.gn1 = GraphNorm(hidden_channels * heads)
        
        self.conv2 = GATv2Conv(hidden_channels * heads, hidden_channels, heads=heads, concat=True, edge_dim=1)
        self.gn2 = GraphNorm(hidden_channels * heads)

        self.conv3 = GATv2Conv(hidden_channels * heads, hidden_channels, heads=heads, concat=True, edge_dim=1)
        self.gn3 = GraphNorm(hidden_channels * heads)
        
        # Final classifier layer
        self.lin = nn.Linear(hidden_channels * heads, out_channels)

    def forward(self, x, edge_index, edge_attr, batch, return_attention=False):
        # Layer 1
        x, att1 = self.conv1(x, edge_index, edge_attr, return_attention_weights=True)
        x = F.leaky_relu(x)
        x = self.gn1(x, batch)
        
        # Layer 2
        x, att2 = self.conv2(x, edge_index, edge_attr, return_attention_weights=True)
        x = F.leaky_relu(x)
        x = self.gn2(x, batch)
        
        # Layer 3
        x, att3 = self.conv3(x, edge_index, edge_attr, return_attention_weights=True)
        x = self.gn3(x, batch)
        
        # Readout layer for graph-level classification
        x = global_mean_pool(x, batch)
        
        # Apply dropout for regularization
        x = F.dropout(x, p=0.5, training=self.training)
        
        # Final classification
        x = self.lin(x)
        
        if return_attention:
            # Return final logits and the attention weights from all layers
            return F.log_softmax(x, dim=1), (att1, att2, att3)
        else:
            return F.log_softmax(x, dim=1)

# --- Instantiate the model and print its summary ---
# in_channels = number of node features (8 frequency bands)
# hidden_channels = an intermediate dimension
# out_channels = number of classes (2: Left/Right)
# heads = number of parallel attention mechanisms
model = GAT(in_channels=data_list[0].num_node_features, 
            hidden_channels=16, 
            out_channels=2, 
            heads=4)

print("\n--- GAT Model Architecture ---")
print(model)
print("----------------------------")

In [ ]:

def bandpass(data: np.ndarray, edges: list[float], sample_rate: float, poles: int = 5):
    sos = sig.butter(poles, edges, 'bandpass', fs=sample_rate, output='sos')
    filtered_data = sig.sosfiltfilt(sos, data,axis=0)
    return filtered_data

def plvfcn(eegData):
    numElectrodes = eegData.shape[1]
    numTimeSteps = eegData.shape[0]
    plvMatrix = np.zeros((numElectrodes, numElectrodes))
    for electrode1 in range(numElectrodes):
        for electrode2 in range(electrode1 + 1, numElectrodes):
            phase1 = np.angle(sig.hilbert(eegData[:, electrode1]))
            phase2 = np.angle(sig.hilbert(eegData[:, electrode2]))
            phase_difference = phase2 - phase1
            plv = np.abs(np.sum(np.exp(1j * phase_difference)) / numTimeSteps)
            plvMatrix[electrode1, electrode2] = plv
            plvMatrix[electrode2, electrode1] = plv
    return plvMatrix

def compute_plv(subject_data):
    idx = ['L', 'R']
    numElectrodes = subject_data['L'][0,0].shape[1]  # Fixed indexing
    
    # Get number of trials for each class
    num_trials_L = subject_data['L'].shape[1]
    num_trials_R = subject_data['R'].shape[1]
    
    plv = {
        'L': np.zeros((numElectrodes, numElectrodes, num_trials_L)), 
        'R': np.zeros((numElectrodes, numElectrodes, num_trials_R))
    }
    
    for field in idx:
        num_trials = subject_data[field].shape[1]
        for j in range(num_trials):
            x = subject_data[field][0, j]
            plv[field][:, :, j] = plvfcn(x)
    
    l, r = plv['L'], plv['R']
    yl = np.zeros((num_trials_L, 1))
    yr = np.ones((num_trials_R, 1))
    img = np.concatenate((l, r), axis=2)
    y = np.concatenate((yl, yr), axis=0)
    y = torch.tensor(y, dtype=torch.long)
    return img, y


def create_graphs(plv, threshold):
    
    graphs = []
    for i in range(plv.shape[2]):
        G = nx.Graph()
        G.add_nodes_from(range(plv.shape[0]))
        for u in range(plv.shape[0]):
            for v in range(plv.shape[0]):
                if u != v and plv[u, v, i] > threshold:
                    G.add_edge(u, v, weight=plv[u, v, i])
        graphs.append(G)
    return graphs



In [ ]:
def load_bci_competition_data(data_dir, subject_number):
    """
    Load BCI Competition IV Dataset 2a and convert to the expected format.
    
    Args:
        data_dir: Directory containing the .mat files
        subject_number: Subject number (1-9)
    
    Returns:
        subject_data: Dictionary with 'L' and 'R' keys containing trial data
    """
    mat_fname = pjoin(data_dir, f'A0{subject_number}T.mat')
    mat_contents = sio.loadmat(mat_fname)
    data = mat_contents['data']
    
    # Initialize lists to store left and right hand trials
    left_trials = []
    right_trials = []
    
    # Process test sessions (indices 3-8)
    for session_idx in range(3, 9):  # Sessions 3-8 contain test data
        session_data = data[0, session_idx][0, 0]
        
        eeg_data = session_data[0]          # Shape: (time_samples, 25_channels)
        trial_positions = session_data[1].flatten()  # Trial start positions
        trial_labels = session_data[2].flatten()     # Trial labels
        
        # Only use first 22 channels (EEG, exclude EOG)
        eeg_data = eeg_data[:, :22]
        
        # Extract trials (each trial is approximately 4 seconds = 1000 samples at 250Hz)
        trial_length = 1000  # 4 seconds * 250 Hz
        
        for i, (start_pos, label) in enumerate(zip(trial_positions, trial_labels)):
            start_sample = int(start_pos)
            end_sample = start_sample + trial_length
            
            # Extract trial data
            if end_sample <= eeg_data.shape[0]:
                trial_data = eeg_data[start_sample:end_sample, :]
                
                # Assign to left or right hand based on label
                if label == 1:  # Left hand
                    left_trials.append(trial_data)
                elif label == 2:  # Right hand
                    right_trials.append(trial_data)
                # Skip feet (3) and tongue (4) for binary classification
    
    # Convert to the expected format
    subject_data = {
        'L': np.empty((1, len(left_trials)), dtype=object),
        'R': np.empty((1, len(right_trials)), dtype=object)
    }
    
    # Fill the arrays
    for i, trial in enumerate(left_trials):
        subject_data['L'][0, i] = trial
    
    for i, trial in enumerate(right_trials):
        subject_data['R'][0, i] = trial
    
    return subject_data

# Test the function with subject 1
data_dir = r'/kaggle/input/bci-competition-iv-data-sets-2a'
test_subject = load_bci_competition_data(data_dir, 1)
print(f"Left hand trials: {test_subject['L'].shape[1]}")
print(f"Right hand trials: {test_subject['R'].shape[1]}")
print(f"Trial shape example: {test_subject['L'][0,0].shape}")

In [ ]:
def aggregate_eeg_data(S1, band): 
    """
    Aggregate EEG data for each class.
    Adapted for BCI Competition IV Dataset 2a format.
    """
    idx = ['L', 'R']
    numElectrodes = S1['L'][0,0].shape[1]  # Should be 22 for this dataset
    
    # Find the maximum trial length (should be consistent at 1000 samples)
    max_sizes = {field: 0 for field in idx}
    for field in idx:
        for i in range(S1[field].shape[1]):
            max_sizes[field] = max(max_sizes[field], S1[field][0, i].shape[0])

    # Initialize arrays to store aggregated EEG data
    l = np.zeros((max_sizes['L'], numElectrodes, S1['L'].shape[1]))
    r = np.zeros((max_sizes['R'], numElectrodes, S1['R'].shape[1]))

    # Loop through each trial
    for i in range(S1['L'].shape[1]):
        x = S1['L'][0, i]  # EEG data for the current trial
        l[:x.shape[0], :, i] = x

    for i in range(S1['R'].shape[1]):
        x = S1['R'][0, i]  # EEG data for the current trial
        r[:x.shape[0], :, i] = x

    # Add frequency band dimension
    l = l[..., np.newaxis]
    l = np.tile(l, (1, 1, 1, len(band)-1))

    r = r[..., np.newaxis]
    r = np.tile(r, (1, 1, 1, len(band)-1))
    
    return l, r

def bandpass1(data: np.ndarray, edges: list[float], sample_rate: float, poles: int = 5):
    sos = sig.butter(poles, edges, 'bandpass', fs=sample_rate, output='sos')
    filtered_data = sig.sosfiltfilt(sos, data)
    return filtered_data

def bandpower(data, low, high, fs=250):
    """Calculate band power using Welch's method"""
    # Define window length (2s)
    win = 2 * fs
    freqs, psd = signal.welch(data, fs, nperseg=min(win, len(data)))
    
    # Find intersecting values in frequency vector
    idx_band = np.logical_and(freqs >= low, freqs <= high)
    
    # Frequency resolution
    freq_res = freqs[1] - freqs[0]
    
    # Compute the absolute power by approximating the area under the curve
    power = simpson(psd[idx_band], dx=freq_res)
    
    return power

def bandpowercalc(l, band, fs):   
    x = np.zeros([l.shape[0], l.shape[3], l.shape[2]])
    for i in range(l.shape[0]):  # node/channel
        for j in range(l.shape[2]):  # trial/sample
            for k in range(l.shape[3]):  # frequency band
                data = l[i, :, j, k]
                low = band[k]
                high = band[k+1]
                x[i, k, j] = bandpower(data, low, high, fs)
    return x


In [ ]:
class GAT(nn.Module):
            def __init__(self, hidden_channels, heads):
                super(GAT, self).__init__()
                
                self.conv1 = GATv2Conv(len(band)-1, hidden_channels, heads=heads, concat=True)  # 8 frequency bands
                self.conv2 = GATv2Conv(hidden_channels * heads, hidden_channels, heads=heads, concat=True)
                self.conv3 = GATv2Conv(hidden_channels * heads, hidden_channels, heads=heads, concat=True)
                
                self.gn1 = GraphNorm(hidden_channels * heads)
                self.gn2 = GraphNorm(hidden_channels * heads)
                self.gn3 = GraphNorm(hidden_channels * heads)
                
                self.lin = nn.Linear(hidden_channels * heads, 2)  # Binary classification
        
            def forward(self, x, edge_index, batch):
                x = self.conv1(x, edge_index)
                x = F.relu(x)
                x = self.gn1(x, batch)
        
                x = self.conv2(x, edge_index)
                x = F.relu(x)
                x = self.gn2(x, batch)
        
                x = self.conv3(x, edge_index)
                x = self.gn3(x, batch)
        
                x = global_mean_pool(x, batch)
                x = F.dropout(x, p=0.50, training=self.training)
                x = self.lin(x)
        
                return x

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


In [ ]:

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# ==============================
# MAIN PIPELINE
# ==============================
data_dir = r'/kaggle/input/bci-competition-iv-data-sets-2a'
subject_numbers = [1, 2, 3, 5, 6, 7, 8, 9]   # Subjects 1-9
subject_results = {}

# Process each subject
for subject_number in tqdm(subject_numbers, desc="Processing Subjects"):
    # Load data using your custom function
    S1 = load_bci_competition_data(data_dir, subject_number)
    
    # Apply bandpass filtering
    fs = 250  # Sampling frequency for BCI Competition IV Dataset 2a
    idx = ['L', 'R']
    for key in idx:
        for i in range(S1[key].shape[1]):
            S1[key][0, i] = bandpass(S1[key][0, i], [8, 30], fs)
    
    # Compute PLV and create graphs
    plv, y = compute_plv(S1)
    threshold = 0
    graphs = create_graphs(plv, threshold)
    numElectrodes = S1['L'][0, 0].shape[1]  # Should be 22
    
    # Create adjacency matrices
    adj = np.zeros([numElectrodes, numElectrodes, len(graphs)])
    for i, G in enumerate(graphs):
        adj[:, :, i] = nx.to_numpy_array(G)
    
    # Create edge indices for PyTorch Geometric
    edge_indices = []
    for i in range(adj.shape[2]):
        source_nodes, target_nodes = [], []
        for row in range(adj.shape[0]):
            for col in range(adj.shape[1]):
                if adj[row, col, i] >= threshold:
                    source_nodes.append(row)
                    target_nodes.append(col)
                else:
                    source_nodes.append(0)
                    target_nodes.append(0)
        edge_index = torch.tensor([source_nodes, target_nodes], dtype=torch.long)
        edge_indices.append(edge_index)

    edge_indices = torch.stack(edge_indices, dim=-1)
    
    # Frequency band analysis
    band = list(range(8, 41, 4))  # [8, 12, 16, 20, 24, 28, 32, 36, 40]
    l, r = aggregate_eeg_data(S1, band)
    l, r = np.transpose(l, [1, 0, 2, 3]), np.transpose(r, [1, 0, 2, 3])
    
    for i in range(l.shape[3]):
        bp = [band[i], band[i+1]]
        for j in range(l.shape[2]):
            l[:, :, j, i] = bandpass1(l[:, :, j, i], bp, sample_rate=fs)
            r[:, :, j, i] = bandpass1(r[:, :, j, i], bp, sample_rate=fs)
    
    l = bandpowercalc(l, band, fs)
    r = bandpowercalc(r, band, fs)

    # Combine left and right hand data
    x = np.concatenate([l, r], axis=2)
    x = torch.tensor(x, dtype=torch.float32)

    # Create PyTorch Geometric data objects
    data_list = []
    for i in range(np.size(adj, 2)):
        data_list.append(Data(x=x[:, :, i], edge_index=edge_indices[:, :, i], y=y[i, 0]))
        
    # Split data into left and right classes
    size = len(data_list)
    idx_split = size // 2
    
    datal = data_list[:idx_split]
    datar = data_list[idx_split:]

    # Interleave left and right trials
    data_list = []
    for i in range(idx_split):
        data_list.extend([datal[i], datar[i]])

    # ==============================
    # K-Fold Cross Validation
    # ==============================
    kf = KFold(n_splits=10, shuffle=True, random_state=42)
    highest_test_accuracies = []
    
    for fold, (train_idx, test_idx) in enumerate(kf.split(data_list)):
        train = [data_list[i] for i in train_idx]
        test = [data_list[i] for i in test_idx]

        torch.manual_seed(12345)
        train_loader = DataLoader(train, batch_size=32, shuffle=False)
        test_loader = DataLoader(test, batch_size=32, shuffle=False)

        model = GAT(hidden_channels=22, heads=3).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
        criterion = torch.nn.CrossEntropyLoss()

        # ==============================
        # TRAIN / TEST FUNCTIONS
        # ==============================
        def train():
            model.train()
            for data in train_loader:
                data = data.to(device)
                out = model(data.x, data.edge_index, data.batch)
                loss = criterion(out, data.y)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        def test(loader):
            model.eval()
            correct = 0
            with torch.no_grad():
                for data in loader:
                    data = data.to(device)
                    out = model(data.x, data.edge_index, data.batch)
                    pred = out.argmax(dim=1)
                    correct += int((pred == data.y).sum())
            return correct / len(loader.dataset)

        # ==============================
        # TRAINING LOOP
        # ==============================
        optimal = [0, 0, 0]
        for epoch in tqdm(range(1, 250), desc=f"Training Subject {subject_number} Fold {fold+1}", leave=False):
            train()
            train_acc = test(train_loader)
            test_acc = test(test_loader)
            av_acc = np.mean([train_acc, test_acc])
            if test_acc > optimal[2]:
                optimal = [av_acc, train_acc, test_acc]

        highest_test_accuracies.append(optimal[2])

    # ==============================
    # RESULTS AGGREGATION
    # ==============================
    meanhigh = np.mean(highest_test_accuracies)
    maxhigh = np.max(highest_test_accuracies)
    minhigh = np.min(highest_test_accuracies)

    subject_results[subject_number] = {
        'mean': meanhigh,
        'max': maxhigh,
        'min': minhigh
    }

    print(f'S{subject_number}: Mean: {meanhigh:.4f}, Max: {maxhigh:.4f}, Min: {minhigh:.4f}')

# ==============================
# SUMMARY AND SAVE RESULTS
# ==============================
print("\nSummary of Results for All Subjects:")
for subject_number, results in subject_results.items():
    print(f'S{subject_number}: Mean: {results["mean"]:.4f}, Max: {results["max"]:.4f}, Min: {results["min"]:.4f}')

with open('BCI_IV_2a_GAT_Results.json', 'w') as json_file:
    json.dump(subject_results, json_file, indent=4)

print("\nResults saved to 'BCI_IV_2a_GAT_Results.json'")

# ==============================
# BOX PLOT OF RESULTS
# ==============================
plt.figure(figsize=(10, 6))
sns.boxplot(data=[[r['mean'] for r in subject_results.values()]], orient='h', color='skyblue')
plt.scatter([r['mean'] for r in subject_results.values()], [0]*len(subject_results), color='red', label='Mean per Subject')
plt.yticks([0], ['Subjects 1–9'])
plt.xlabel('Accuracy')
plt.title('Distribution of Mean Test Accuracies (BCI Competition IV 2a - GAT Model)')
plt.legend()
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Define the data
data = {
    "S1": {"Mean": 0.7086, "Max": 0.7857, "Min": 0.6000},
    "S2": {"Mean": 0.7438, "Max": 0.8667, "Min": 0.6000},
    "S3": {"Mean": 0.7638, "Max": 0.8667, "Min": 0.6429},
    "S5": {"Mean": 0.7500, "Max": 0.9333, "Min": 0.5333},
    "S6": {"Mean": 0.6743, "Max": 0.7857, "Min": 0.6000},
    "S7": {"Mean": 0.7857, "Max": 0.8667, "Min": 0.6667},
    "S8": {"Mean": 0.7357, "Max": 0.9333, "Min": 0.6429},
    "S9": {"Mean": 0.8262, "Max": 0.9333, "Min": 0.6667},
}

# Create a DataFrame
df = pd.DataFrame(data).T

# Generate synthetic data around mean for box plot visualization
synthetic_data = {}
for subject, row in df.iterrows():
    synthetic_data[subject] = [row["Min"], row["Mean"], row["Max"], row["Mean"], row["Max"]]

# Convert synthetic data to DataFrame for boxplot
synthetic_df = pd.DataFrame(synthetic_data)

# Create box plot
plt.figure(figsize=(10, 6))
plt.boxplot(synthetic_df.values, labels=synthetic_df.columns, patch_artist=True)
plt.title("Box Plot for Each Subject")
plt.ylabel("Values")
plt.grid(True, linestyle="--", alpha=0.6)
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Prepare data for boxplot
subject_labels = []
accuracies = []

for subject_number, results in subject_results.items():
    subject_labels.append(f'S{subject_number}')
    accuracies.append(results['mean'])

# Boxplot of mean accuracies per subject
plt.figure(figsize=(10, 6))
sns.boxplot(data=[accuracies], orient='h', width=0.4, color='skyblue')
plt.scatter(accuracies, [0]*len(accuracies), color='red', label='Mean Accuracy per Subject')
plt.yticks([0], ['Subjects 1–9'])
plt.xlabel('Accuracy')
plt.title('Distribution of Mean Test Accuracies (BCI Competition IV 2a - GAT Model)')
plt.legend()
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.show()


In [ ]:
# ==========================================================
# EEG MI Classification using PLV-based Graph Attention Network
# Combined version: Full pipeline with Kaggle-friendly structure
# ==========================================================

import os
import numpy as np
import torch
import torch.nn.functional as F
import torch.nn as nn
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATv2Conv, GraphNorm, global_mean_pool

from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score
from tqdm.notebook import tqdm
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import networkx as nx
import json

# ==========================================================
# 1. SETUP
# ==========================================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

data_dir = '/kaggle/input/bci-competition-iv-data-sets-2a'
subject_numbers = [1, 2, 3, 5, 6, 7, 8, 9]  # Skip 4 if missing
fs = 250
freq_band = [8, 30]

# ==========================================================
# 2. HELPER FUNCTIONS
# ==========================================================

def bandpass(signal, band, fs):
    """Apply bandpass filter to EEG signal."""
    from scipy.signal import butter, filtfilt
    low, high = band
    b, a = butter(4, [low / (fs / 2), high / (fs / 2)], btype='band')
    return filtfilt(b, a, signal)

def compute_plv(data_dict):
    """Compute PLV (Phase Locking Value) between all EEG channels."""
    from scipy.signal import hilbert
    L = data_dict['L']
    R = data_dict['R']
    n_trials = len(L) + len(R)
    n_channels = L[0].shape[0]

    plv_mats = []
    labels = []
    for trials, label in [(L, 0), (R, 1)]:
        for trial in trials:
            analytic = hilbert(trial)
            phase = np.angle(analytic)
            plv = np.abs(np.exp(1j * phase) @ np.exp(-1j * phase).T) / phase.shape[1]
            np.fill_diagonal(plv, 0)
            plv_mats.append(plv)
            labels.append(label)
    return np.array(plv_mats), np.array(labels)

def create_graphs(plv_array, threshold=0.0):
    """Create NetworkX graphs from PLV matrices."""
    graphs = []
    for mat in plv_array:
        G = nx.Graph()
        n_nodes = mat.shape[0]
        for i in range(n_nodes):
            for j in range(i+1, n_nodes):
                if mat[i, j] > threshold:
                    G.add_edge(i, j, weight=mat[i, j])
        graphs.append(G)
    return graphs

def bandpower(signal, fs):
    """Compute band power using signal variance."""
    return np.mean(signal**2, axis=-1)

def create_pyg_datalist(X, y, graphs, band_features):
    """Create PyTorch Geometric data list."""
    data_list = []
    for i in range(len(graphs)):
        edge_index = torch.tensor(list(graphs[i].edges)).t().contiguous()
        if edge_index.numel() == 0:
            continue
        x = torch.tensor(band_features[i], dtype=torch.float32)
        data = Data(x=x, edge_index=edge_index, y=torch.tensor(y[i]).long())
        data_list.append(data)
    return data_list


# ==========================================================
# 3. DEFINE THE GAT MODEL
# ==========================================================
class GAT(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, heads):
        super(GAT, self).__init__()
        self.conv1 = GATv2Conv(in_channels, hidden_channels, heads=heads)
        self.gn1 = GraphNorm(hidden_channels * heads)
        self.conv2 = GATv2Conv(hidden_channels * heads, hidden_channels, heads=heads)
        self.gn2 = GraphNorm(hidden_channels * heads)
        self.lin = nn.Linear(hidden_channels * heads, out_channels)

    def forward(self, x, edge_index, batch=None):
        x = self.conv1(x, edge_index)
        x = F.relu(self.gn1(x, batch))
        x = self.conv2(x, edge_index)
        x = F.relu(self.gn2(x, batch))
        x = global_mean_pool(x, batch)
        x = F.dropout(x, p=0.5, training=self.training)
        return F.log_softmax(self.lin(x), dim=1)


# ==========================================================
# 4. MAIN ANALYSIS LOOP
# ==========================================================
all_subject_results = {}

for subject_id in subject_numbers:
    print(f"\n{'='*25} Processing Subject {subject_id} {'='*25}")

    # a) Load and preprocess
    raw_data = load_bci_competition_data(data_dir, subject_id)
    for key in raw_data:
        for i in range(len(raw_data[key])):
            raw_data[key][i] = bandpass(raw_data[key][i], freq_band, fs)

    # b) Compute PLV and construct graphs
    plv_array, y = compute_plv(raw_data)
    graphs = create_graphs(plv_array, threshold=0.0)
    num_nodes = plv_array.shape[1]

    # c) Extract bandpower features for each trial (as node features)
    band_features = []
    for trials, label in [(raw_data['L'], 0), (raw_data['R'], 1)]:
        for trial in trials:
            bp = bandpower(trial, fs)
            band_features.append(bp.reshape(num_nodes, 1))  # node feature dim = 1

    band_features = np.array(band_features, dtype=object)

    # d) K-Fold Cross Validation
    kf = KFold(n_splits=10, shuffle=True, random_state=42)
    subject_fold_accuracies = []

    for fold, (train_idx, test_idx) in enumerate(kf.split(band_features, y)):
        print(f"Subject {subject_id}, Fold {fold+1}/10")

        # Prepare datasets
        train_data_list = create_pyg_datalist(band_features[train_idx],
                                              y[train_idx],
                                              [graphs[i] for i in train_idx],
                                              band_features[train_idx])
        test_data_list = create_pyg_datalist(band_features[test_idx],
                                             y[test_idx],
                                             [graphs[i] for i in test_idx],
                                             band_features[test_idx])

        if len(train_data_list) == 0 or len(test_data_list) == 0:
            continue

        train_loader = DataLoader(train_data_list, batch_size=16, shuffle=True)
        test_loader = DataLoader(test_data_list, batch_size=16, shuffle=False)

        # Initialize model
        model = GAT(in_channels=1, hidden_channels=22, out_channels=2, heads=3).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=5e-4)
        criterion = nn.NLLLoss()

        # Train for 150 epochs
        for epoch in range(1, 151):
            model.train()
            for data in train_loader:
                data = data.to(device)
                optimizer.zero_grad()
                out = model(data.x, data.edge_index, data.batch)
                loss = criterion(out, data.y)
                loss.backward()
                optimizer.step()

        # Evaluate
        model.eval()
        preds, true = [], []
        with torch.no_grad():
            for data in test_loader:
                data = data.to(device)
                out = model(data.x, data.edge_index, data.batch)
                pred = out.argmax(dim=1)
                preds.extend(pred.cpu().numpy())
                true.extend(data.y.cpu().numpy())

        acc = accuracy_score(true, preds)
        print(f"Fold {fold+1} Accuracy: {acc:.4f}")
        subject_fold_accuracies.append(acc)

    # e) Subject summary
    mean_acc = np.mean(subject_fold_accuracies)
    std_acc = np.std(subject_fold_accuracies)
    all_subject_results[f"S{subject_id}"] = subject_fold_accuracies
    print(f"Subject {subject_id}: {mean_acc:.4f} ± {std_acc:.4f}")

# ==========================================================
# 5. VISUALIZATION
# ==========================================================
print(f"\n{'='*20} Final Summary {'='*20}")
results_df = pd.DataFrame(all_subject_results)
print("Mean Accuracy per Subject:\n", results_df.mean())

plt.figure(figsize=(12, 8))
sns.boxplot(data=results_df, palette="viridis")
sns.stripplot(data=results_df, color=".25", size=4)
plt.title("PLV-GAT Model Performance Across Subjects (10-Fold CV)", fontsize=16)
plt.ylabel("Classification Accuracy", fontsize=12)
plt.xlabel("Subject ID", fontsize=12)
plt.axhline(0.5, ls="--", color="red", label="Chance Level (50%)")
plt.legend()
plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.ylim(0.3, 1.0)
plt.show()

# ==========================================================
# 6. SAVE RESULTS
# ==========================================================
with open("BCI_IV_2a_GAT_Results.json", "w") as f:
    json.dump(all_subject_results, f, indent=4)

print("\n✅ Results saved to 'BCI_IV_2a_GAT_Results.json'")
